# 12 - Binary Two-Phase Model: Error Analysis

**Purpose:** Analyse errors (FN, FP, high-confidence errors) of the binary cancer-risk model
on the test set using the Youden J threshold = 0.4219.

**Rules:**
- No training. No model retraining.
- Grad-CAM intentionally deferred to File 14.
- No image/manifest/split modifications.

> **Medical note:** This model estimates cancer-risk probability for screening support.
> It is not a clinical diagnosis.

---


## Section 0 - Imports and Environment

In [1]:
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torchvision

CUDA_AVAILABLE = torch.cuda.is_available()
try:
    DEVICE   = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "N/A"
except AssertionError:
    DEVICE, GPU_NAME, CUDA_AVAILABLE = torch.device("cpu"), "N/A", False

print(f"torch       : {torch.__version__}")
print(f"CUDA        : {CUDA_AVAILABLE}  GPU: {GPU_NAME}")


torch       : 2.6.0+cu124
CUDA        : True  GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Section 1 - Paths and Configuration

In [2]:
OUTPUT_ROOT = Path(r"C:\SKIN CANCER v2\pipe output")
EVAL_DIR    = OUTPUT_ROOT / "pytorch_binary_2phase_evaluation"
ERR_DIR     = OUTPUT_ROOT / "pytorch_binary_2phase_error_analysis"
PREPROC_DIR = OUTPUT_ROOT / "preprocessing"

YOUDEN_THRESH     = 0.4219
HIGH_RECALL_THRESH = 0.1700
BINARY_INDEX = {"NV": 0, "MEL": 1, "BCC": 1}
BINARY_NAMES = ["non_cancer", "cancer_risk"]

# High-confidence error cutoffs (top-N most confident errors)
N_HIGH_CONF = 30
GRID_COLS   = 6
IMG_CELL    = (2.4, 2.8)   # inches per cell (width, height)

print("Config loaded.")
print(f"  Main threshold (Youden J) : {YOUDEN_THRESH}")
print(f"  High-recall threshold     : {HIGH_RECALL_THRESH}")
print(f"  Eval dir                  : {EVAL_DIR}")
print(f"  Error dir                 : {ERR_DIR}")


Config loaded.
  Main threshold (Youden J) : 0.4219
  High-recall threshold     : 0.17
  Eval dir                  : C:\SKIN CANCER v2\pipe output\pytorch_binary_2phase_evaluation
  Error dir                 : C:\SKIN CANCER v2\pipe output\pytorch_binary_2phase_error_analysis


## Section 2 - Create Output Folder

In [3]:
ERR_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {ERR_DIR}")


Ready: C:\SKIN CANCER v2\pipe output\pytorch_binary_2phase_error_analysis


## Section 3 - Load File 11 Test Predictions

In [4]:
pred_path = EVAL_DIR / "binary_2phase_test_predictions.csv"
if not pred_path.exists():
    raise FileNotFoundError(
        f"Test predictions not found: {pred_path}\n"
        "Run 11_pytorch_binary_2phase_final_evaluation.ipynb first."
    )

pred_df = pd.read_csv(pred_path, low_memory=False)
print(f"Loaded {pred_path.name}: {len(pred_df):,} rows")
print(f"Columns: {list(pred_df.columns)}")

# Also load preprocessed manifest to get extra metadata columns
test_manifest = pd.read_csv(
    PREPROC_DIR / "test_manifest_preprocessed.csv", low_memory=False)
print(f"Loaded test manifest: {len(test_manifest):,} rows")


Loaded binary_2phase_test_predictions.csv: 3,045 rows
Columns: ['preprocessed_full_path', 'original_full_path', 'final_authoritative_label', 'binary_label', 'canonical_match_id', 'lesion_id', 'split', 'binary_true_label', 'cancer_risk_probability', 'predicted_binary_default']
Loaded test manifest: 3,045 rows


## Section 4 - Validate Columns and Apply Threshold

In [5]:
required_cols = [
    "preprocessed_full_path", "final_authoritative_label",
    "binary_true_label", "cancer_risk_probability",
]
missing = [c for c in required_cols if c not in pred_df.columns]
if missing:
    raise ValueError(f"Missing required columns in predictions CSV: {missing}")
print("Required columns present.")

# Ensure binary_true_label is numeric
pred_df["binary_true_label"] = pd.to_numeric(pred_df["binary_true_label"], errors="coerce").astype(int)

# Apply threshold if predicted label column is missing
if "predicted_binary_label" not in pred_df.columns:
    pred_df["predicted_binary_label"] = (
        pred_df["cancer_risk_probability"] >= YOUDEN_THRESH).astype(int)
    print(f"Created predicted_binary_label using threshold {YOUDEN_THRESH}")
else:
    # Recompute at Youden threshold regardless
    pred_df["predicted_binary_youden"] = (
        pred_df["cancer_risk_probability"] >= YOUDEN_THRESH).astype(int)
    print("Using existing predicted_binary_label; also added predicted_binary_youden column.")

pred_df["threshold_used"] = YOUDEN_THRESH

# Merge extra metadata from manifest (vignette info, lesion_id, etc.)
merge_cols = [c for c in [
    "preprocessed_full_path", "original_full_path",
    "canonical_match_id", "lesion_id",
    "vignette_crop_applied", "crop_method", "crop_confidence_status",
] if c in test_manifest.columns]
extra = test_manifest[merge_cols].copy()
pred_df = pred_df.merge(extra, on="preprocessed_full_path", how="left", suffixes=("", "_manifest"))
print(f"Merged metadata. Final columns: {list(pred_df.columns)}")

# Summary counts
pred_lbl = pred_df["predicted_binary_youden"] if "predicted_binary_youden" in pred_df.columns else pred_df["predicted_binary_label"]
true_lbl  = pred_df["binary_true_label"]
TP = int(((true_lbl == 1) & (pred_lbl == 1)).sum())
TN = int(((true_lbl == 0) & (pred_lbl == 0)).sum())
FP = int(((true_lbl == 0) & (pred_lbl == 1)).sum())
FN = int(((true_lbl == 1) & (pred_lbl == 0)).sum())
print(f"\nAt threshold={YOUDEN_THRESH}:")
print(f"  TP={TP}  TN={TN}  FP={FP}  FN={FN}")
print(f"  Cancer recall = {TP/(TP+FN):.4f}  Specificity = {TN/(TN+FP):.4f}")


Required columns present.
Created predicted_binary_label using threshold 0.4219
Merged metadata. Final columns: ['preprocessed_full_path', 'original_full_path', 'final_authoritative_label', 'binary_label', 'canonical_match_id', 'lesion_id', 'split', 'binary_true_label', 'cancer_risk_probability', 'predicted_binary_default', 'predicted_binary_label', 'threshold_used', 'original_full_path_manifest', 'canonical_match_id_manifest', 'lesion_id_manifest', 'vignette_crop_applied', 'crop_method', 'crop_confidence_status']

At threshold=0.4219:
  TP=976  TN=1608  FP=286  FN=175
  Cancer recall = 0.8480  Specificity = 0.8490


## Section 5 - Classify Errors

In [6]:
PRED_COL = "predicted_binary_youden" if "predicted_binary_youden" in pred_df.columns else "predicted_binary_label"

fn_mask  = (pred_df["binary_true_label"] == 1) & (pred_df[PRED_COL] == 0)
fp_mask  = (pred_df["binary_true_label"] == 0) & (pred_df[PRED_COL] == 1)
mel_fn   = fn_mask & (pred_df["final_authoritative_label"] == "MEL")
bcc_fn   = fn_mask & (pred_df["final_authoritative_label"] == "BCC")

export_cols = [c for c in [
    "preprocessed_full_path", "original_full_path",
    "final_authoritative_label", "binary_true_label",
    "cancer_risk_probability", PRED_COL,
    "threshold_used", "canonical_match_id", "lesion_id",
    "vignette_crop_applied", "crop_method", "crop_confidence_status",
] if c in pred_df.columns]

fn_df   = pred_df[fn_mask][export_cols].copy()
fp_df   = pred_df[fp_mask][export_cols].copy()
mel_fn_df = pred_df[mel_fn][export_cols].copy()
bcc_fn_df = pred_df[bcc_fn][export_cols].copy()

print(f"False negatives (missed cancer): {len(fn_df):,}")
print(f"  Missed MEL                   : {len(mel_fn_df):,}")
print(f"  Missed BCC                   : {len(bcc_fn_df):,}")
print(f"False positives (over-triggered): {len(fp_df):,}")


False negatives (missed cancer): 175
  Missed MEL                   : 148
  Missed BCC                   : 27
False positives (over-triggered): 286


## Section 6 - High-Confidence Errors

In [7]:
# High-confidence FN: model was most confident they were non_cancer (lowest prob)
hc_fn_df = fn_df.nsmallest(N_HIGH_CONF, "cancer_risk_probability").copy()
hc_fn_df["error_type"] = "high_conf_false_negative"

# High-confidence FP: model was most confident they were cancer_risk (highest prob)
hc_fp_df = fp_df.nlargest(N_HIGH_CONF, "cancer_risk_probability").copy()
hc_fp_df["error_type"] = "high_conf_false_positive"

hc_combined = pd.concat([hc_fn_df, hc_fp_df], ignore_index=True)

print(f"High-confidence false negatives (top-{N_HIGH_CONF} lowest prob): {len(hc_fn_df)}")
print(f"  Prob range: {hc_fn_df['cancer_risk_probability'].min():.4f} – "
      f"{hc_fn_df['cancer_risk_probability'].max():.4f}")
print(f"High-confidence false positives (top-{N_HIGH_CONF} highest prob): {len(hc_fp_df)}")
print(f"  Prob range: {hc_fp_df['cancer_risk_probability'].min():.4f} – "
      f"{hc_fp_df['cancer_risk_probability'].max():.4f}")


High-confidence false negatives (top-30 lowest prob): 30
  Prob range: 0.0002 – 0.0292
High-confidence false positives (top-30 highest prob): 30
  Prob range: 0.9883 – 1.0000


## Section 7 - Save Error CSVs

In [8]:
fn_path      = ERR_DIR / "binary_2phase_false_negatives.csv"
fp_path      = ERR_DIR / "binary_2phase_false_positives.csv"
hcfn_path    = ERR_DIR / "binary_2phase_high_confidence_false_negatives.csv"
hcfp_path    = ERR_DIR / "binary_2phase_high_confidence_false_positives.csv"

fn_df.to_csv(fn_path, index=False)
fp_df.to_csv(fp_path, index=False)
hc_fn_df.to_csv(hcfn_path, index=False)
hc_fp_df.to_csv(hcfp_path, index=False)

for p, df in [(fn_path, fn_df), (fp_path, fp_df),
              (hcfn_path, hc_fn_df), (hcfp_path, hc_fp_df)]:
    print(f"  Saved {p.name:<55} ({len(df):>4} rows)")


  Saved binary_2phase_false_negatives.csv                       ( 175 rows)
  Saved binary_2phase_false_positives.csv                       ( 286 rows)
  Saved binary_2phase_high_confidence_false_negatives.csv       (  30 rows)
  Saved binary_2phase_high_confidence_false_positives.csv       (  30 rows)


## Section 8 - Image Grids

In [9]:
def load_img(path):
    try:
        return Image.open(str(path)).convert("RGB")
    except Exception:
        return None


def make_grid(df, title, save_path, prob_col="cancer_risk_probability",
              label_col="final_authoritative_label", n_max=30,
              bg_color="mistyrose"):
    rows_to_show = df.head(n_max)
    n = len(rows_to_show)
    if n == 0:
        print(f"  Skipping {save_path.name} — no samples to show")
        return
    n_rows = (n + GRID_COLS - 1) // GRID_COLS
    fig, axes = plt.subplots(n_rows, GRID_COLS,
                              figsize=(GRID_COLS * IMG_CELL[0], n_rows * IMG_CELL[1]))
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax in axes_flat:
        ax.axis("off")
        ax.set_facecolor(bg_color)

    for i, (_, row) in enumerate(rows_to_show.iterrows()):
        ax  = axes_flat[i]
        img = load_img(row["preprocessed_full_path"])
        prob = float(row.get(prob_col, -1))
        lbl  = str(row.get(label_col, "?"))
        if img is not None:
            ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"{lbl}\np={prob:.3f}", fontsize=7, pad=2)

    plt.suptitle(title, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close()
    print(f"  Saved {save_path.name}")


print("Generating image grids...")

make_grid(mel_fn_df, f"Missed MELanomas (FN) — threshold={YOUDEN_THRESH}",
          ERR_DIR / "binary_2phase_missed_melanoma_grid.png",
          n_max=30, bg_color="mistyrose")

make_grid(bcc_fn_df, f"Missed BCC (FN) — threshold={YOUDEN_THRESH}",
          ERR_DIR / "binary_2phase_missed_bcc_grid.png",
          n_max=30, bg_color="lightyellow")

make_grid(fp_df, f"False Positives (NV predicted cancer_risk) — threshold={YOUDEN_THRESH}",
          ERR_DIR / "binary_2phase_false_positive_grid.png",
          n_max=30, bg_color="lightcyan")

make_grid(hc_combined,
          f"High-Confidence Errors (top {N_HIGH_CONF} FN + top {N_HIGH_CONF} FP)",
          ERR_DIR / "binary_2phase_high_confidence_error_grid.png",
          n_max=60, bg_color="lavender")

print("Grid generation complete.")


Generating image grids...
  Saved binary_2phase_missed_melanoma_grid.png
  Saved binary_2phase_missed_bcc_grid.png
  Saved binary_2phase_false_positive_grid.png
  Saved binary_2phase_high_confidence_error_grid.png
Grid generation complete.


## Section 9 - Error Summary CSV

In [10]:
from sklearn.metrics import roc_auc_score, average_precision_score

summary_df = pd.DataFrame([{
    "threshold_name":           "youden_j",
    "threshold":                YOUDEN_THRESH,
    "total_test_rows":          len(pred_df),
    "total_cancer_risk":        int((pred_df["binary_true_label"]==1).sum()),
    "total_non_cancer":         int((pred_df["binary_true_label"]==0).sum()),
    "TP": TP, "TN": TN, "FP": FP, "FN": FN,
    "cancer_recall":            round(TP/max(TP+FN,1), 5),
    "specificity":              round(TN/max(TN+FP,1), 5),
    "false_negatives":          FN,
    "false_positives":          FP,
    "missed_mel":               len(mel_fn_df),
    "missed_bcc":               len(bcc_fn_df),
    "hc_false_negatives":       len(hc_fn_df),
    "hc_false_positives":       len(hc_fp_df),
    "gradcam_deferred_to":      "14_real_image_binary_inference_gradcam.ipynb",
    "training_performed":       False,
    "images_modified":          False,
}])
sum_path = ERR_DIR / "binary_2phase_error_summary.csv"
summary_df.to_csv(sum_path, index=False)
print(f"Saved {sum_path.name}")
from IPython.display import display
display(summary_df.T)


Saved binary_2phase_error_summary.csv


,0
threshold_name,youden_j
threshold,0.4219
total_test_rows,3045
total_cancer_risk,1151
total_non_cancer,1894
TP,976
TN,1608
FP,286
FN,175
cancer_recall,0.84796


## Section 10 - Output File Verification

In [11]:
required_files = [
    ERR_DIR / "binary_2phase_error_summary.csv",
    ERR_DIR / "binary_2phase_false_negatives.csv",
    ERR_DIR / "binary_2phase_false_positives.csv",
    ERR_DIR / "binary_2phase_high_confidence_false_negatives.csv",
    ERR_DIR / "binary_2phase_high_confidence_false_positives.csv",
    ERR_DIR / "binary_2phase_missed_melanoma_grid.png",
    ERR_DIR / "binary_2phase_missed_bcc_grid.png",
    ERR_DIR / "binary_2phase_false_positive_grid.png",
    ERR_DIR / "binary_2phase_high_confidence_error_grid.png",
]
print("Output file verification:")
all_ok = True
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<55} {size:>10,} bytes")
    if not exists: all_ok = False
print("\nAll files present." if all_ok else "\nWARNING: missing files.")


Output file verification:
  [OK] binary_2phase_error_summary.csv                                403 bytes
  [OK] binary_2phase_false_negatives.csv                           32,485 bytes
  [OK] binary_2phase_false_positives.csv                           51,838 bytes
  [OK] binary_2phase_high_confidence_false_negatives.csv            6,607 bytes
  [OK] binary_2phase_high_confidence_false_positives.csv            6,356 bytes
  [OK] binary_2phase_missed_melanoma_grid.png                   1,765,618 bytes
  [OK] binary_2phase_missed_bcc_grid.png                        2,065,316 bytes
  [OK] binary_2phase_false_positive_grid.png                    1,796,725 bytes
  [OK] binary_2phase_high_confidence_error_grid.png             4,228,465 bytes

All files present.


## Section 11 - Final Summary (Copy-Paste Ready)

In [12]:
print("=" * 70)
print("  12_binary_2phase_error_analysis -- FINAL SUMMARY")
print("=" * 70)
print(f"\n 1. Threshold used (Youden J) : {YOUDEN_THRESH}")
print(f" 2. Total test rows            : {len(pred_df):,}")
print(f" 3. False negatives (FN)       : {FN}  (missed cancer)")
print(f" 4. False positives (FP)       : {FP}  (over-triggered)")
print(f" 5. Missed MEL                 : {len(mel_fn_df)}")
print(f" 6. Missed BCC                 : {len(bcc_fn_df)}")
print(f" 7. High-confidence FN         : {len(hc_fn_df)}  (lowest probability)")
print(f" 8. High-confidence FP         : {len(hc_fp_df)}  (highest probability)")
print(f"\n 9. Grid files saved:")
grids = ["binary_2phase_missed_melanoma_grid.png",
         "binary_2phase_missed_bcc_grid.png",
         "binary_2phase_false_positive_grid.png",
         "binary_2phase_high_confidence_error_grid.png"]
for g in grids:
    p = ERR_DIR / g
    print(f"      {'OK' if p.exists() else 'MISSING'} {g}")
print(f"\n10. Output file verification:")
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    print(f"      [{'OK' if exists else 'MISSING'}] {p.name:<55} {size:>8,} bytes")
print(f"\n11. Training performed         : False")
print(f"12. Images/manifests modified  : False")
print(f"13. Grad-CAM                   : Intentionally deferred to File 14")
print(f"\n> Medical note: This model estimates cancer-risk probability for")
print(f"> screening support. It is not a clinical diagnosis.")
print("=" * 70)


  12_binary_2phase_error_analysis -- FINAL SUMMARY

 1. Threshold used (Youden J) : 0.4219
 2. Total test rows            : 3,045
 3. False negatives (FN)       : 175  (missed cancer)
 4. False positives (FP)       : 286  (over-triggered)
 5. Missed MEL                 : 148
 6. Missed BCC                 : 27
 7. High-confidence FN         : 30  (lowest probability)
 8. High-confidence FP         : 30  (highest probability)

 9. Grid files saved:
      OK binary_2phase_missed_melanoma_grid.png
      OK binary_2phase_missed_bcc_grid.png
      OK binary_2phase_false_positive_grid.png
      OK binary_2phase_high_confidence_error_grid.png

10. Output file verification:
      [OK] binary_2phase_error_summary.csv                              403 bytes
      [OK] binary_2phase_false_negatives.csv                         32,485 bytes
      [OK] binary_2phase_false_positives.csv                         51,838 bytes
      [OK] binary_2phase_high_confidence_false_negatives.csv          6,607 byt

## Section 12 - Completion Summary

**12_binary_2phase_error_analysis is complete.**

**What was accomplished:**
- Test predictions from File 11 loaded and validated.
- False negatives (missed cancers), false positives identified at Youden J threshold.
- High-confidence error cases identified (lowest prob FN, highest prob FP).
- Image grids saved for qualitative inspection.
- Error summary CSV saved.

**What was deliberately deferred:**
- Grad-CAM / saliency maps → File 14
- Clinical calibration analysis
